### 🏗️ Notebook: Load Silver Layer
**Target:** `restaurant_project.silver.rest_sales`

#### 🎯 Purpose
* **Integration**: Union CSV and JSON sources into a single unified sales table.
* **Cleaning**: Remove whitespaces (Trim) and filter out inconsistent or corrupted records.
* **Accuracy**: Re-calculate `total_amount` to ensure mathematical consistency.
* **Identification**: Generate a unique **Surrogate Key (SK)** for every record.

#### 🛠️ Key Transformations
1. **Filtering**: Remove records with `price <= 0` or `discount > price`.
2. **Formatting**: Apply `TRIM` to columns: `item_name`, `category`, `branch`, `order_type`, and `payment_method`.
3. **Calculation**: Apply formula: `ROUND((price * quantity) * (1 - discount), 2)`.
4. **Metadata**: Add `silver_load_timestamp` to track processing time.
3
###🚀 Execution Strategy
* **Format**: Delta Table.
* **Mode**: `Overwrite` with `overwriteSchema` enabled.

In [0]:
from pyspark.sql import functions as F  
from pyspark.sql.functions import expr 
import time


In [0]:
from pyspark.sql import functions as F  
from pyspark.sql.functions import expr 
import time
# 1. source tables bronze
source_csv_table = "restaurant_project.bronze.raw_sales_csv"
source_json_table = "restaurant_project.bronze.raw_sales_json"

# 2. target table silver
target_silver_table = "restaurant_project.silver.rest_sales"

def load_silver():
    start_time = time.time()
    print('================================================')
    print('Starting Silver Layer Transformation Operations')
    print('================================================')

    try:
        # 1. Read from Bronze (Union both sources if needed, or process combined)
        df_csv = spark.table(source_csv_table)
        df_json = spark.table(source_json_table)
        df_rest_sales = df_csv.unionByName(df_json, allowMissingColumns=True)

        print(f">> Initial Row Count: {df_rest_sales.count()}")

        # 2. Transformations & Cleaning
        df_rest_sales = df_rest_sales \
            .withColumn("sales_skey", F.monotonically_increasing_id()) \
            .withColumn("item_name", F.trim(F.col("item_name"))) \
            .withColumn("category", F.trim(F.col("category"))) \
            .withColumn("branch", F.trim(F.col("branch"))) \
            .withColumn("order_type", F.trim(F.col("order_type"))) \
            .withColumn("payment_method", F.trim(F.col("payment_method"))) \
            .filter(
                (F.col("price") > 0) & 
                (F.col("discount") <= F.col("price"))
            ) \
            .withColumn("silver_load_timestamp", F.current_timestamp())

        # 3. Handling total_amount 
        df_rest_sales = df_rest_sales.withColumn(
            "total_amount", 
            expr("ROUND((price * quantity) * (1.0 - discount), 2)")
         ).withColumn("silver_load_timestamp", F.current_timestamp())
          
        

        # 4. Write to silver schema
        df_rest_sales.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(target_silver_table)

        duration = __builtins__.round(time.time() - start_time, 2)
        print(f">> SUCCESS:") 
        print(f">> Rows Processed & Saved: {df_rest_sales.count()}")
        print(f">> Target Table: {target_silver_table}")
        print(f">> Silver Load Duration: {duration} seconds")
        print('==========================================')

    except Exception as e:
        print('==========================================')
        print('ERROR OCCURED DURING LOADING SILVER LAYER')
        print(f'Error Message: {str(e)}')
        print('==========================================')

# Start the process
load_silver()


        
        
    